# 00 Data Engineering

            This notebook prepares the raw Bank Marketing dataset for modeling and deployment. It creates two cleaned datasets: one production-safe dataset that excludes `duration`, and one diagnostic dataset that keeps `duration` for comparison.

            The production decision is important because `duration` is only known after a customer call ends. It can help explain outcomes, but it should not be used when scoring customers before outreach.


## Load Project Paths and Raw Data

            Paths are defined relative to the project root so the notebook can run from either the `notebooks/` folder or the root directory. The raw source file contains 11,162 customer records and 17 variables.


In [1]:
# Set project paths and load the raw bank marketing CSV.

from pathlib import Path
import sys
import pandas as pd
import logging

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'bank.csv'
CLEANED_NO_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_bank_data.csv'
CLEANED_WITH_DURATION = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_bank_data_with_duration.csv'

bank_data = pd.read_csv(RAW_DATA)
bank_data.shape


(11162, 17)

The shape confirms that the local source file has 11,162 rows and 17 columns. At this stage the data is still unmodified and includes both `duration` and `pdays`.


In [2]:
# Preview the first records to confirm schema and raw value formats.

bank_data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


The source rows contain numeric variables such as `age`, `balance`, `day`, `duration`, `campaign`, `pdays`, and `previous`, along with categorical variables such as `job`, `marital`, `education`, `contact`, `month`, and `poutcome`.


## Check Data Types, Missing Values, and Cardinality

            This data-quality check verifies whether any column requires imputation, whether categorical variables are suitable for one-hot encoding, and whether the target is complete.


In [3]:
# Summarize data types, missing values, and unique values by column.

pd.DataFrame({
    'dtype': bank_data.dtypes.astype(str),
    'missing': bank_data.isna().sum(),
    'unique_values': bank_data.nunique()
})


,dtype,missing,unique_values
age,int64,0,76
job,object,0,12
marital,object,0,3
education,object,0,4
default,object,0,2
balance,int64,0,3805
housing,object,0,2
loan,object,0,2
contact,object,0,3
day,int64,0,31


The dataset has no missing values. Imputation remains part of the reusable pipeline for robustness, but this specific source file does not require missing-value replacement. Values such as `unknown` are kept as valid categories because they contain campaign information rather than true null values.


## Inspect Target Balance

            The target distribution is checked before modeling because class balance affects model selection and evaluation.


In [4]:
# Count and normalize the term-deposit subscription target.

bank_data['deposit'].value_counts(), bank_data['deposit'].value_counts(normalize=True)


(no     5873
 yes    5289
 Name: deposit, dtype: int64,
 no     0.52616
 yes    0.47384
 Name: deposit, dtype: float64)

The target is moderately balanced: about 52.6% `no` and 47.4% `yes`. Since the classes are close enough, no resampling is applied. Model evaluation focuses on accuracy, sensitivity, specificity, balanced accuracy, F1, and ROC AUC.


## Contact Channel Check

            A normalized crosstab is used to compare subscription rates across contact channels.


In [5]:
# Compare deposit rates by contact channel.

pd.crosstab(bank_data['contact'], bank_data['deposit'], normalize='index')


deposit,no,yes
contact,,
cellular,0.456727,0.543273
telephone,0.496124,0.503876
unknown,0.774084,0.225916


Contact channel has a visible relationship with subscription. Customers contacted by `cellular` subscribe more often than customers with an `unknown` contact type, so `contact` is retained as a categorical predictor.


## Create Production and Diagnostic Datasets

            Two cleaned datasets are created:

            - `cleaned_bank_data.csv`: production-safe data with `duration` and `pdays` removed.
            - `cleaned_bank_data_with_duration.csv`: diagnostic comparison data that keeps `duration` but removes `pdays`.

            `pdays` is removed in both cases because the value `-1` dominates the column, indicating that many customers were not previously contacted.


In [6]:
# Run the reusable cleaning pipeline for both modeling scenarios.

from src.data.run_processing import process_data

logging.disable(logging.INFO)   # suppress INFO logs

no_duration = process_data(RAW_DATA, CLEANED_NO_DURATION, keep_duration=False)
with_duration = process_data(RAW_DATA, CLEANED_WITH_DURATION, keep_duration=True)

logging.disable(logging.NOTSET)  # re-enable logging

no_duration.shape, with_duration.shape


((11162, 15), (11162, 16))

The no-duration dataset has 15 columns, while the with-duration dataset has 16 columns. The only difference between the two modeling scenarios is the presence of `duration`.


## Final Cleaned Data Preview

            The cleaned production dataset is reviewed before moving into feature engineering. At this point, `deposit` is encoded as `0` for `no` and `1` for `yes`, while categorical predictors remain as text for the preprocessing pipeline.


In [ ]:
# Inspect the cleaned production dataset.

no_duration.head()s


,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1,0,unknown,1
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1,0,unknown,1
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1,0,unknown,1
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,1,0,unknown,1
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,2,0,unknown,1
